# Tender Analysis System

## 1. Import Libraries and Configuration

In [19]:
import os
import re
import json
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Tuple, Optional
import warnings
from datetime import datetime
import logging
from functools import lru_cache
import hashlib
import time
from pathlib import Path
import pickle

warnings.filterwarnings('ignore')

# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('tender_analysis_v8.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# ============================================================================
# CONFIGURATION
# ============================================================================

class TenderAnalysisConfig:
    """Centralized configuration for the tender analysis system."""
    
    # PDF Processing
    PDF_PATH = "DST_URA_E_GRYKSHIT_II (2).pdf"
    CHUNK_SIZE = 1000
    CHUNK_OVERLAP = 150
    
    # Thresholds
    SEMANTIC_SEARCH_THRESHOLD = 0.40
    CRITERIA_MATCH_THRESHOLD = 0.45
    REQUIREMENT_EXTRACTION_THRESHOLD = 0.40
    
    # Model
    EMBEDDING_MODEL = 'paraphrase-multilingual-mpnet-base-v2'
    BATCH_SIZE = 32
    
    # Output
    OUTPUT_DIR = 'outputs_v8'
    VECTOR_STORE_FILE = 'tender_vector_store.pkl'
    OUTPUT_REPORT = 'tender_extraction_report.json'
    OUTPUT_TEXT = 'tender_extraction_report.txt'

config = TenderAnalysisConfig()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

print("✓ Configuration loaded successfully")
print(f"✓ Output directory: {config.OUTPUT_DIR}")
print(f"✓ Embedding model: {config.EMBEDDING_MODEL}")

✓ Configuration loaded successfully
✓ Output directory: outputs_v8
✓ Embedding model: paraphrase-multilingual-mpnet-base-v2


## 2. Define Tender Criteria Framework

27 tender criteria organized by category for comprehensive evaluation.

In [20]:
# ============================================================================
# TENDER CRITERIA DEFINITIONS
# ============================================================================

TENDER_CRITERIA = {
    'Kritere Administrative': [
        'Kompania duhet të jetë e regjistruar ligjërisht (NIPT, QKB, licenca etj.)',
        'Të mos ketë detyrime tatimore apo sigurimesh shoqërore të pashlyera',
        'Të mos ketë qenë e dënuar për shkelje ligjore apo mashtrim tenderash',
        'Të ketë vërtetim nga gjykata që nuk është në proces falimentimi ose likuidimi',
        'Dokumentacioni të jetë plotësisht i nënshkruar, vulosur dhe në afat'
    ],
    'Kritere Teknike': [
        'Ofruesi duhet të përmbushë specifikimet teknike të përcaktuara në dokumentet e tenderit',
        'Duhet të ofrohet përshkrim teknik i produkteve/shërbimeve (katalogë, broshura, certifikata)',
        'Duhet të ketë eksperiencë të ngjashme me projektet e kërkuara (referenca nga kontrata të mëparshme)',
        'Pajisjet, produktet ose shërbimet duhet të jenë në përputhje me standardet ISO, EN, CE ose të tjera të përcaktuara',
        'Personeli i angazhuar duhet të jetë i kualifikuar dhe i certifikuar profesionalisht'
    ],
    'Kritere Financiare': [
        'Kompania duhet të ketë gjendje financiare të qëndrueshme (bilancet e 2-3 viteve të fundit)',
        'Duhet të paraqesë xhiro minimale vjetore në raport me vlerën e tenderit',
        'Duhet të ketë kapacitet financiar për të mbuluar kostot e projektit (likuiditete, kredi, garanci bankare)',
        'Garancia e ofertës (zakonisht 2-5% e vlerës së kontratës) duhet të paraqitet me afat vlefshmërie të përshtatshëm'
    ],
    'Kritere të Cilësisë dhe Sigurisë': [
        'Ofertuesi duhet të ketë sisteme të menaxhimit të cilësisë (p.sh. ISO 9001)',
        'Nëse aplikohet, duhet të ketë certifikime për siguri në punë (p.sh. ISO 45001)',
        'Duhet të përmbushë rregullat për mbrojtjen e mjedisit (ISO 14001)',
        'Duhet të ofrojë garanci për produktet/shërbimet për një periudhë të caktuar pas dorëzimit'
    ],
    'Kritere të Vlerësimit të Ofertës': [
        'Çmimi më i ulët (në tendera të thjeshtë)',
        'Raporti më i mirë cilësi-çmim (në tendera kompleksë)',
        'Vlerësimi përfshin: Çmimin total, Cilësinë teknike, Afatet e dorëzimit, Garancitë e ofruara, Shërbimin pas-shitjes'
    ],
    'Kritere për Përgjegjshmëri Sociale & Ligjore': [
        'Respektim i ligjeve të punës dhe sigurimesh shoqërore',
        'Barazi gjinore dhe mosdiskriminim',
        'Përdorim i qëndrueshëm i burimeve dhe mbrojtje e mjedisit'
    ],
    'Kritere të Veçanta': [
        'Për tendera ndërtimi: leje ndërtimi, inxhinierë të licencuar, makina të certifikuara',
        'Për IT: prova koncepti (POC), performancë teknike, siguri e të dhënave (GDPR, ISO 27001)',
        'Për shërbime konsulence: CV e ekspertëve, metodologjia e punës, afatet e zbatimit'
    ]
}

print(f"✓ Loaded {sum(len(v) for v in TENDER_CRITERIA.values())} tender criteria")
print(f"✓ {len(TENDER_CRITERIA)} criteria categories defined")

✓ Loaded 27 tender criteria
✓ 7 criteria categories defined


## 3. Utility and Helper Functions

Core utility functions for text processing and page mapping.

In [21]:
# ============================================================================
# PDF EXTRACTION WITH PyMuPDF
# ============================================================================

try:
    import fitz  # PyMuPDF
    PYMUPDF_AVAILABLE = True
    logger.info("PyMuPDF loaded successfully")
except ImportError:
    PYMUPDF_AVAILABLE = False
    logger.error("PyMuPDF not available - install with: pip install pymupdf")
    raise ImportError("PyMuPDF required for PDF processing")

def extract_text_from_pdf(pdf_path: str) -> Tuple[str, Dict]:
    """Extract with page mapping using PyMuPDF."""
    try:
        doc = fitz.open(pdf_path)
        text = ""
        page_map = {}
        current_pos = 0
        
        for page_num in range(len(doc)):
            page_text = doc[page_num].get_text()
            page_marker = f"\n--- PAGE {page_num + 1} ---\n"
            text += page_marker + page_text
            
            page_map[page_num + 1] = {
                'start': current_pos,
                'end': current_pos + len(page_marker) + len(page_text),
                'length': len(page_text)
            }
            current_pos += len(page_marker) + len(page_text)
        
        doc.close()
        logger.info(f"Extracted {len(text)} characters from {len(page_map)} pages")
        return text, page_map
    except Exception as e:
        logger.error(f"PDF extraction failed: {e}")
        return "", {}

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def clean_text(text: str) -> str:
    """Clean text from artifacts."""
    text = re.sub(r'\s*\|+\s*', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'_+', '', text)
    return text.strip()

def find_page_number(position: int, page_map: Dict) -> int:
    """Find page number from character position."""
    for page_num, info in page_map.items():
        if info['start'] <= position <= info['end']:
            return page_num
    return 1

print("✓ Utility functions loaded")

2025-11-23 15:01:44,367 - INFO - PyMuPDF loaded successfully


✓ Utility functions loaded


## 4. Load Embedding Model

Initialize the multilingual sentence transformer model for semantic search.

In [22]:
# ============================================================================
# EMBEDDING MODEL
# ============================================================================

@lru_cache(maxsize=1)
def load_embedding_model(model_name: str = config.EMBEDDING_MODEL):
    """Load and cache the embedding model."""
    logger.info(f"Loading model: {model_name}")
    model = SentenceTransformer(model_name)
    logger.info("Model loaded successfully")
    return model

print("Loading embedding model (this may take a minute on first run)...")
model = load_embedding_model()
print(f"✓ Model loaded: {config.EMBEDDING_MODEL}")

2025-11-23 15:01:47,168 - INFO - Loading model: paraphrase-multilingual-mpnet-base-v2
2025-11-23 15:01:47,180 - INFO - Load pretrained SentenceTransformer: paraphrase-multilingual-mpnet-base-v2


Loading embedding model (this may take a minute on first run)...


2025-11-23 15:01:50,339 - INFO - Use pytorch device_name: cpu
2025-11-23 15:01:50,347 - INFO - Model loaded successfully


✓ Model loaded: paraphrase-multilingual-mpnet-base-v2


## 5. Text Chunking and Vector Store Creation

Create intelligent text chunks with page tracking and generate embeddings.

In [5]:
# ============================================================================
# TEXT CHUNKING WITH PAGE TRACKING
# ============================================================================

def smart_chunk_text(text: str, page_map: Dict, chunk_size: int = config.CHUNK_SIZE, 
                    overlap: int = config.CHUNK_OVERLAP) -> List[Dict]:
    """Intelligently split text with page tracking."""
    if len(text.strip()) < 50:
        return []
    
    chunks = []
    section_pattern = r'\n(?:#{1,3}\s+|[A-ZËÇSH]{3,}[A-ZËÇSH\s]{0,100}:|(?:\d+\.)+\s+[A-ZËÇSH])'
    sections = re.split(section_pattern, text)
    
    for i, section in enumerate(sections):
        if len(section.strip()) < 50:
            continue
        
        section_start = text.find(section)
        section_page = find_page_number(section_start, page_map) if section_start >= 0 else 1
        
        lines = section.split('\n')
        title = lines[0].strip()[:100] if lines else f"Section {i+1}"
        content = '\n'.join(lines[1:]) if len(lines) > 1 else section
        
        sentences = re.split(r'[.!?]\s+', content)
        current_chunk = []
        current_length = 0
        
        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue
            
            if current_length + len(sentence) >= chunk_size and current_chunk:
                chunk_text = ' '.join(current_chunk)
                chunks.append({
                    'text': chunk_text,
                    'section': title,
                    'page': section_page,
                    'length': len(chunk_text),
                    'chunk_id': len(chunks)
                })
                
                overlap_words = ' '.join(current_chunk[-overlap:]) if len(current_chunk) > overlap else ' '.join(current_chunk)
                current_chunk = [overlap_words] if overlap_words else []
                current_length = len(overlap_words)
            
            current_chunk.append(sentence)
            current_length += len(sentence)
        
        if current_chunk and current_length > 100:
            chunks.append({
                'text': ' '.join(current_chunk),
                'section': title,
                'page': section_page,
                'length': len(' '.join(current_chunk)),
                'chunk_id': len(chunks)
            })
    
    logger.info(f"Created {len(chunks)} chunks across {len(page_map)} pages")
    return chunks

# ============================================================================
# VECTOR STORE
# ============================================================================

def create_vector_store(chunks_data: List[Dict], model: SentenceTransformer) -> Dict:
    """Create vector store with embeddings."""
    if len(chunks_data) == 0:
        return {'chunks': [], 'embeddings': np.array([]), 'metadata': []}
    
    chunks = [c['text'] for c in chunks_data]
    
    logger.info(f"Generating embeddings for {len(chunks)} chunks")
    embeddings = model.encode(
        chunks,
        show_progress_bar=True,
        batch_size=config.BATCH_SIZE,
        convert_to_numpy=True
    )
    
    vector_store = {
        'chunks': chunks,
        'embeddings': embeddings,
        'metadata': chunks_data,
        'created_at': datetime.now().isoformat()
    }
    
    return vector_store

print("✓ Chunking and vector store functions loaded")

✓ Chunking and vector store functions loaded


## 6. Semantic Search with Similarity Scoring

Perform semantic search with normalized cosine similarity (0-1 scale).

In [6]:
# ============================================================================
# SEMANTIC SEARCH WITH SIMILARITY SCORES
# ============================================================================

def semantic_search(query: str, vector_store: Dict, model: SentenceTransformer, 
                   top_k: int = 10, min_score: float = config.SEMANTIC_SEARCH_THRESHOLD) -> List[Dict]:
    """Perform semantic search with normalized cosine similarity."""
    if len(vector_store['chunks']) == 0:
        return []
    
    try:
        query_embedding = model.encode([query], show_progress_bar=False)[0]
        
        # Normalize for true 0-1 cosine similarity
        query_norm = query_embedding / (np.linalg.norm(query_embedding) + 1e-8)
        doc_norms = vector_store['embeddings'] / (np.linalg.norm(vector_store['embeddings'], axis=1, keepdims=True) + 1e-8)
        
        similarities = np.dot(doc_norms, query_norm)
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        results = []
        for i in top_indices:
            if similarities[i] >= min_score:
                results.append({
                    'text': vector_store['chunks'][i],
                    'similarity': float(similarities[i]),
                    'similarity_percent': float(similarities[i] * 100),
                    'index': i,
                    'chunk_id': vector_store['metadata'][i].get('chunk_id', i),
                    'page': vector_store['metadata'][i].get('page', 'N/A'),
                    'section': vector_store['metadata'][i].get('section', 'N/A')
                })
        
        return results
    except Exception as e:
        logger.error(f"Search failed: {e}")
        return []

print("✓ Semantic search function loaded")

✓ Semantic search function loaded


## 7. Extract Deadline Information

Extract tender submission deadlines in Albanian format (dd.mm.yyyy).

In [7]:
# ============================================================================
# DEADLINE EXTRACTION
# ============================================================================

def extract_deadline(text: str) -> Dict:
    """Extract deadline with Albanian format support."""
    deadline_info = {
        'date': '',
        'time': '',
        'full': '',
        'found': False
    }
    
    # Search in first 30000 chars
    header = text[:30000]
    
    # Patterns for Albanian tender deadlines
    patterns = [
        r'Afati\s+i\s+fundit\s+për\s+paraqitjen.*?(\d{1,2}\.\d{1,2}\.\d{4})\s+Ora[:\s]+(\d{1,2}:\d{2})',
        r'(\d{1,2}\.\d{1,2}\.\d{4})\s+Ora[:\s]+(\d{1,2}:\d{2})',
        r'Data:\s*(\d{1,2}\.\d{1,2}\.\d{4})\s+\(d/m/v\)\s+[Oo]ra[:\s]+(\d{1,2}:\d{2})',
        r'(\d{1,2}[./]\d{1,2}[./]\d{4})\s+[Oo]ra[:\s]+(\d{1,2}:\d{2})'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, header, re.IGNORECASE)
        if match:
            deadline_info['date'] = match.group(1).replace('/', '.')
            deadline_info['time'] = match.group(2) if len(match.groups()) > 1 else ''
            deadline_info['full'] = f"{deadline_info['date']} Ora {deadline_info['time']}"
            deadline_info['found'] = True
            logger.info(f"Found deadline: {deadline_info['full']}")
            break
    
    return deadline_info

print("✓ Deadline extraction function loaded")

✓ Deadline extraction function loaded


## 8. Extract Machinery Requirements from Tables

Extract machinery requirements using table detection and pattern matching.

In [8]:
# ============================================================================
# MACHINERY EXTRACTION FROM TABLE
# ============================================================================

def extract_machinery_from_table(text: str) -> List[Dict]:
    """Extract machinery from table-like regions with better patterns."""
    
    def clean_text_local(s):
        s = re.sub(r'\s*\|+\s*', ' ', s)
        s = re.sub(r'\s+', ' ', s)
        s = re.sub(r'_+', '', s)
        return s.strip()
    
    def extract_quantity_from_text(text_segment: str) -> int:
        """Extract quantity from various formats."""
        patterns = [
            r'(\d+)\s*(?:copë|cop|cope|njësi)',
            r'(?:copë|cop|cope|njësi)\s*[:=]?\s*(\d+)',
            r'Sasia\s*[:=]?\s*(\d+)',
            r'Nr\.\s*[:=]?\s*(\d+)'
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text_segment, re.IGNORECASE)
            if match:
                return int(match.group(1))
        
        numbers = re.findall(r'\b(\d+)\b', text_segment)
        if numbers:
            return int(numbers[-1])
        
        return 1

    machinery_list = []
    machinery_keywords = [
        'kamion', 'ekskavator', 'buldozer', 'vinç', 'pompë', 'autobot',
        'fadrome', 'rulo', 'greider', 'asfalto', 'kamionçin', 'eskavator',
        'autobetonier', 'mikser', 'cisternë', 'grejder', 'kamioncin',
        'autovinç', 'vibrator', 'kompresor', 'gjenerator', 'betoniere',
        'saldatrice', 'minieskavator', 'minifadrome', 'loader', 'bobcat'
    ]

    # Find the machinery table section
    table_patterns = [
        r'(?:Lloji\s+i\s+makinerive|Lista\s+e\s+makinerive|Makineritë).*?(?:Nr\.|Sasia|Copë)(.*?)(?:Shënim|Totali|Personeli|Stafi|\n\n\n)',
        r'Nr\.?\s+Lloji\s+i\s+Makinerisë.*?Sasia(.*?)(?:\n\n|\Z)',
        r'MAKINER[IË].*?(?:Sasia|Copë)(.*?)(?:PERSON|STAF|\n\n\n)',
    ]
    
    table_text = None
    for pattern in table_patterns:
        match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        if match:
            table_text = match.group(1) if len(match.groups()) > 0 else match.group(0)
            logger.info(f"Found machinery table section ({len(table_text)} chars)")
            break
    
    if not table_text:
        logger.info("No machinery table found")
        return []
    
    lines = table_text.split('\n')
    seen_items = set()
    row_num = 0
    
    for idx, line in enumerate(lines):
        line_clean = clean_text_local(line)
        if len(line_clean) < 3:
            continue
        
        if any(header in line_clean.lower() for header in ['lloji', 'sasia', 'makinerive', 'nr.']):
            continue
        
        found_machinery = False
        for keyword in machinery_keywords:
            if keyword in line_clean.lower():
                found_machinery = True
                break
        
        if not found_machinery:
            continue
        
        row_num += 1
        
        # Extract machinery
        match = re.match(r'^(\d+)[\s\.]+(.*?)(?:copë|cop|njësi)?\s*[:=]?\s*(\d+)', line_clean, re.IGNORECASE)
        if match:
            machine_name = clean_text_local(match.group(2))
            quantity = int(match.group(3))
        else:
            match = re.match(r'^(.*?)(?:copë|cop|njësi)?\s*[:=]?\s*(\d+)', line_clean, re.IGNORECASE)
            if match:
                machine_name = clean_text_local(match.group(1))
                quantity = int(match.group(2))
            else:
                continue
        
        machine_name = re.sub(r'\d+\s*$', '', machine_name).strip()
        if len(machine_name) < 3 or machine_name.isdigit():
            continue
        
        spec_match = re.search(r'(me|mbi|deri|nga)\s+(\d+[\s\w]+)', line_clean, re.IGNORECASE)
        specs = clean_text_local(spec_match.group(0)) if spec_match else ''
        
        machine_key = f"{machine_name.lower()}-{quantity}"
        if machine_key in seen_items:
            continue
        seen_items.add(machine_key)
        
        machinery_list.append({
            'type': machine_name.title(),
            'quantity': quantity,
            'specifications': specs,
            'category': 'makineritë',
            'row_number': str(row_num)
        })
        
        logger.info(f"Extracted: {machine_name} | Qty: {quantity}")
    
    logger.info(f"Total machinery items found: {len(machinery_list)}")
    return machinery_list

print("✓ Machinery extraction function loaded")

✓ Machinery extraction function loaded


## 9. Extract Staff Requirements

Direct pattern matching for staff requirements without semantic dependency.

In [10]:
# ============================================================================
# IMPROVED STAFF EXTRACTION
# ============================================================================

def extract_staff_smart_v2(text: str, vector_store: Dict, 
                          model: SentenceTransformer) -> List[Dict]:
    """Extract staff requirements using direct text pattern matching."""
    logger.info("Extracting staff requirements using direct pattern matching")
    
    combined = text
    logger.info(f"Searching in {len(combined)} characters")
    
    staff_list = []
    seen_positions = set()
    
    patterns = [
        r'^\s*1\s*\(\s*(?:nje|një)\s*\)\s+([Ii]nxhinier\s+[A-Za-zëçËÇ\s]+?)(?:\s+\(|$)',
        r'^\s*1\s*\(\s*(?:nje|është)\s*\)\s+([Ii]nxhinier\s+[A-Za-zëçËÇ]+)',
        r'^\s*1\s*\(\s*(?:nje|është)\s*\)\s+([Pp]unonjës\s+[A-Za-zëçËÇ\s]+?)(?:\s*$|\s+)',
        r'^\s*1\s*\(\s*(?:nje|është)\s*\)\s+([Ss]pecialist\s+[A-Za-zëçËÇ\s]+?)(?:\s*$|\s+)',
        r'^\s*1\s*\(\s*(?:nje|është)\s*\)\s+([A-Za-zëçËÇ\s]{6,60})',
        r'^\s*[Ii]nxhinier\s+(?:Ndërtimi|Ndërtim|Strukturist|Hidroteknik|Elektrik|Gjeolog|Mjedisi|Topograf|Mekanik)',
        r'^\s*[Pp]unonjës\s+[A-Za-zëçËÇ\s]+',
        r'^\s*[Dd]rejtues\s+[Pp]unim',
        r'^\s*[Ss]upervisor',
    ]
    
    lines = combined.split('\n')
    logger.info(f"Processing {len(lines)} lines")
    
    for line_idx, line in enumerate(lines):
        line_stripped = line.strip()
        
        if len(line_stripped) < 4:
            continue
        
        for pattern in patterns:
            match = re.search(pattern, line_stripped, re.IGNORECASE | re.MULTILINE)
            
            if match:
                try:
                    if len(match.groups()) > 0:
                        position = match.group(1).strip() if match.group(1) else line_stripped.strip()
                    else:
                        position = match.group(0).strip()
                    
                    position = re.sub(r'\s+', ' ', position).strip()
                    position = position.strip('|()').strip()
                    position = re.sub(r'\s*\([^)]*profili[^)]*\)\s*', ' ', position, flags=re.IGNORECASE)
                    position = position.strip()
                    
                    if len(position) < 4 or position.isdigit():
                        continue
                    
                    quantity = 1
                    experience = 0
                    
                    normalized = position.lower().strip()
                    if normalized in seen_positions:
                        continue
                    
                    seen_positions.add(normalized)
                    
                    staff_list.append({
                        'position': position,
                        'quantity': quantity,
                        'experience_years': experience,
                        'category': 'staf'
                    })
                    
                    logger.info(f"Found staff: {quantity}x {position}")
                    break
                    
                except Exception as e:
                    logger.warning(f"Error parsing line '{line_stripped}': {e}")
                    continue
    
    logger.info(f"Total staff positions found: {len(staff_list)}")
    return staff_list

print("✓ Staff extraction function loaded")

✓ Staff extraction function loaded


## 10. Extract Certification Requirements

Extract ISO standards, CE certifications, and professional licenses.

In [11]:
# ============================================================================
# IMPROVED CERTIFICATION EXTRACTION
# ============================================================================

def extract_certifications_smart_v2(text: str, vector_store: Dict, 
                                   model: SentenceTransformer) -> List[str]:
    """Extract certification requirements with deduplication."""
    logger.info("Extracting certification requirements using smart v2")
    
    queries = [
        "certifikata standarde ISO sistemi menaxhimit cilësisë",
        "certifikatë CE akreditim vërtetim",
        "urdhri inxhinierëve licencë profesionale regjistrim",
        "ISO 9001 ISO 14001 ISO 45001",
        "certifikim ndërkombëtar standard europian"
    ]
    
    relevant_chunks = []
    seen_chunks = set()
    
    for query in queries:
        results = semantic_search(query, vector_store, model, top_k=8, min_score=0.15)
        logger.info(f"Query '{query}': Found {len(results)} chunks")
        for chunk in results:
            chunk_hash = hashlib.md5(chunk['text'].encode()).hexdigest()
            if chunk_hash not in seen_chunks:
                relevant_chunks.append(chunk)
                seen_chunks.add(chunk_hash)
    
    if not relevant_chunks:
        logger.warning("No relevant chunks found - searching in full text")
        combined = text
    else:
        combined = '\n'.join([c['text'] for c in relevant_chunks])
    
    certifications = []
    seen_certs = set()
    
    # ISO standards
    iso_patterns = [
        r'ISO\s*\d+(?::\d+)?(?:\s*:\s*\d+)?',
        r'ISO-\d+',
        r'EN\s*ISO\s*\d+(?::\d+)?'
    ]
    
    for pattern in iso_patterns:
        iso_matches = re.findall(pattern, combined, re.IGNORECASE)
        for match in iso_matches:
            normalized = match.upper().strip()
            if normalized not in seen_certs:
                certifications.append(normalized)
                seen_certs.add(normalized)
    
    # Other certifications
    cert_patterns = [
        (r'\bCE\b', 'Certifikatë CE'),
        (r'Urdhri\s+(?:i\s+)?Inxhinier[ëa]?', 'Urdhri i Inxhinierëve'),
        (r'[Ll]icenc[ëa]\s+profesionale', 'Licencë Profesionale'),
        (r'OHSAS\s+18001', 'OHSAS 18001'),
        (r'EN\s+\d{3,5}', 'EN Standards'),
        (r'Certifikat(?:ë|a)\s+(?:e\s+)?Cilësisë', 'Certifikatë Cilësisë'),
        (r'Vërtetim\s+(?:e\s+)?Konformiteti', 'Vërtetim Konformiteti'),
    ]
    
    for pattern, cert_name in cert_patterns:
        if re.search(pattern, combined, re.IGNORECASE):
            if cert_name not in seen_certs:
                certifications.append(cert_name)
                seen_certs.add(cert_name)
                logger.info(f"Found: {cert_name}")
    
    logger.info(f"Total certifications found: {len(certifications)}")
    return certifications

print("✓ Certification extraction function loaded")

✓ Certification extraction function loaded


## 11. Match Tender Criteria with Similarity Scores

Match all 27 tender criteria against extracted document.

In [12]:
# ============================================================================
# CRITERIA MATCHING WITH SIMILARITY
# ============================================================================

def match_criteria_with_similarity(vector_store: Dict, model: SentenceTransformer) -> Dict:
    """Match all tender criteria with similarity scores."""
    logger.info("Matching tender criteria with similarity scores")
    
    criteria_results = {}
    
    for category, criteria_list in TENDER_CRITERIA.items():
        logger.info(f"Processing category: {category}")
        category_results = []
        
        for criterion in criteria_list:
            # Search for this criterion
            results = semantic_search(criterion, vector_store, model, top_k=3, min_score=config.CRITERIA_MATCH_THRESHOLD)
            
            if results:
                top_match = results[0]
                category_results.append({
                    'criterion': criterion,
                    'matched': True,
                    'similarity_score': round(top_match['similarity'], 3),
                    'similarity_percent': round(top_match['similarity'] * 100, 1),
                    'chunk_id': top_match['chunk_id'],
                    'page': top_match['page'],
                    'matched_text': top_match['text'][:200] + '...' if len(top_match['text']) > 200 else top_match['text']
                })
            else:
                category_results.append({
                    'criterion': criterion,
                    'matched': False,
                    'similarity_score': 0.0,
                    'similarity_percent': 0.0,
                    'chunk_id': None,
                    'page': None,
                    'matched_text': ''
                })
        
        criteria_results[category] = category_results
    
    # Calculate statistics
    total_criteria = sum(len(cat) for cat in TENDER_CRITERIA.values())
    matched_criteria = sum(1 for cat in criteria_results.values() for c in cat if c['matched'])
    
    criteria_results['_statistics'] = {
        'total_criteria': total_criteria,
        'matched_criteria': matched_criteria,
        'match_percentage': round(matched_criteria / total_criteria * 100, 1) if total_criteria > 0 else 0
    }
    
    logger.info(f"Matched {matched_criteria}/{total_criteria} criteria ({criteria_results['_statistics']['match_percentage']}%)")
    
    return criteria_results

print("✓ Criteria matching function loaded")

✓ Criteria matching function loaded


## 12. Tender Participation Evaluation Engine

Comprehensive scoring and recommendation system.

In [13]:
# ============================================================================
# TENDER DECISION & EVALUATION LOGIC
# ============================================================================

def evaluate_tender_participation(extraction_data: Dict, company_resources: Dict = None) -> Dict:
    """Evaluate if company should participate in tender."""
    logger.info("Evaluating tender participation suitability")
    
    evaluation = {
        'overall_score': 0,
        'recommendation': 'NO',
        'reasons': [],
        'machinery_match': 0,
        'staff_match': 0,
        'certifications_match': 0,
        'criteria_match': 0,
        'deadline_feasibility': 'UNKNOWN',
        'risk_factors': [],
        'opportunities': []
    }
    
    # Parse deadline
    deadline_info = extraction_data.get('deadline', {})
    if deadline_info.get('found'):
        try:
            deadline_str = deadline_info['date']  # Format: DD.MM.YYYY
            parts = deadline_str.split('.')
            if len(parts) == 3:
                deadline = datetime(int(parts[2]), int(parts[1]), int(parts[0]))
                days_left = (deadline - datetime.now()).days
                
                if days_left < 7:
                    evaluation['deadline_feasibility'] = 'CRITICAL'
                    evaluation['risk_factors'].append(f"Very tight deadline: {days_left} days")
                elif days_left < 14:
                    evaluation['deadline_feasibility'] = 'URGENT'
                    evaluation['risk_factors'].append(f"Tight deadline: {days_left} days")
                elif days_left < 30:
                    evaluation['deadline_feasibility'] = 'REASONABLE'
                else:
                    evaluation['deadline_feasibility'] = 'COMFORTABLE'
                    evaluation['opportunities'].append(f"Good time to prepare: {days_left} days")
        except Exception as e:
            logger.warning(f"Could not parse deadline: {e}")
    
    # Evaluate machinery
    machinery = extraction_data.get('machinery', [])
    if machinery:
        evaluation['machinery_match'] = min(100, len(machinery) * 10)
        if evaluation['machinery_match'] >= 80:
            evaluation['opportunities'].append(f"Good machinery coverage: {len(machinery)} items")
        else:
            evaluation['risk_factors'].append(f"Limited machinery: {len(machinery)} items")
    else:
        evaluation['risk_factors'].append("No machinery identified")
    
    # Evaluate staff
    staff = extraction_data.get('staff', [])
    if staff:
        evaluation['staff_match'] = min(100, len(staff) * 15)
        if evaluation['staff_match'] >= 80:
            evaluation['opportunities'].append(f"Adequate staff: {len(staff)} positions")
        else:
            evaluation['risk_factors'].append(f"Specialized staff needed: {len(staff)}")
    else:
        evaluation['risk_factors'].append("No staff identified")
    
    # Evaluate certifications
    certifications = extraction_data.get('certifications', [])
    if certifications:
        evaluation['certifications_match'] = min(100, len(certifications) * 20)
        evaluation['reasons'].append(f"Certifications: {', '.join(certifications[:3])}")
        if len(certifications) > 5:
            evaluation['risk_factors'].append(f"Many certs required: {len(certifications)}")
    else:
        evaluation['opportunities'].append("No special certs needed")
    
    # Evaluate criteria
    criteria = extraction_data.get('criteria_matching', {})
    stats = criteria.get('_statistics', {})
    criteria_match_pct = stats.get('match_percentage', 0)
    evaluation['criteria_match'] = criteria_match_pct
    
    if criteria_match_pct >= 85:
        evaluation['opportunities'].append(f"Strong alignment: {criteria_match_pct}%")
    elif criteria_match_pct >= 70:
        evaluation['reasons'].append(f"Moderate alignment: {criteria_match_pct}%")
    else:
        evaluation['risk_factors'].append(f"Weak alignment: {criteria_match_pct}%")
    
    # Calculate overall score
    weights = {
        'machinery': 0.25,
        'staff': 0.25,
        'certifications': 0.20,
        'criteria': 0.30
    }
    
    evaluation['overall_score'] = (
        evaluation['machinery_match'] * weights['machinery'] +
        evaluation['staff_match'] * weights['staff'] +
        evaluation['certifications_match'] * weights['certifications'] +
        evaluation['criteria_match'] * weights['criteria']
    )
    
    # Make recommendation
    if evaluation['overall_score'] >= 75 and evaluation['deadline_feasibility'] != 'CRITICAL':
        evaluation['recommendation'] = 'YES'
    elif evaluation['overall_score'] >= 60 and len(evaluation['risk_factors']) <= 2:
        evaluation['recommendation'] = 'MAYBE'
    else:
        evaluation['recommendation'] = 'NO'
    
    logger.info(f"Evaluation complete - Score: {evaluation['overall_score']:.1f}%, Recommendation: {evaluation['recommendation']}")
    return evaluation

print("✓ Evaluation engine loaded")

✓ Evaluation engine loaded


## 13. Generate Output Reports

Create multiple report formats (JSON, Text, Excel, HTML).

In [14]:
# ============================================================================
# OUTPUT GENERATION
# ============================================================================

def generate_output_report(extraction_data: Dict, output_dir: str):
    """Generate comprehensive output reports."""
    logger.info("Generating output reports")
    
    # JSON
    json_path = os.path.join(output_dir, config.OUTPUT_REPORT)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(extraction_data, f, ensure_ascii=False, indent=2)
    logger.info(f"JSON saved: {json_path}")
    
    # TEXT
    text_path = os.path.join(output_dir, config.OUTPUT_TEXT)
    with open(text_path, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write("TENDER EXTRACTION REPORT\n")
        f.write("=" * 80 + "\n\n")
        
        deadline = extraction_data.get('deadline', {})
        f.write("DEADLINE:\n")
        f.write(f"  Date: {deadline.get('date', 'N/A')}\n")
        f.write(f"  Time: {deadline.get('time', 'N/A')}\n\n")
        
        f.write("MACHINERY:\n")
        for item in extraction_data.get('machinery', []):
            f.write(f"  - {item['quantity']}x {item['type']}\n")
        f.write("\n")
        
        f.write("STAFF:\n")
        for item in extraction_data.get('staff', []):
            f.write(f"  - {item['quantity']}x {item['position']}\n")
        f.write("\n")
        
        f.write("CERTIFICATIONS:\n")
        for cert in extraction_data.get('certifications', []):
            f.write(f"  - {cert}\n")
        f.write("\n")
        
        f.write("=" * 80 + "\n")
    logger.info(f"TEXT saved: {text_path}")
    
    # EXCEL
    try:
        excel_path = os.path.join(output_dir, 'tender_extraction_report.xlsx')
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            pd.DataFrame(extraction_data.get('machinery', [])).to_excel(writer, sheet_name='Machinery', index=False)
            pd.DataFrame(extraction_data.get('staff', [])).to_excel(writer, sheet_name='Staff', index=False)
            pd.DataFrame({'Certification': extraction_data.get('certifications', [])}).to_excel(writer, sheet_name='Certifications', index=False)
        logger.info(f"EXCEL saved: {excel_path}")
    except Exception as e:
        logger.warning(f"Excel generation failed: {e}")

def generate_tender_decision_report(evaluation: Dict, extraction_data: Dict, output_dir: str):
    """Generate decision report."""
    logger.info("Generating decision report")
    
    report_path = os.path.join(output_dir, 'tender_decision_report.txt')
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write("TENDER DECISION REPORT\n")
        f.write("=" * 80 + "\n\n")
        
        f.write(f"Recommendation: {evaluation['recommendation']}\n")
        f.write(f"Overall Score: {evaluation['overall_score']:.1f}%\n\n")
        
        f.write("Scores:\n")
        f.write(f"  Machinery: {evaluation['machinery_match']:.1f}%\n")
        f.write(f"  Staff: {evaluation['staff_match']:.1f}%\n")
        f.write(f"  Certifications: {evaluation['certifications_match']:.1f}%\n")
        f.write(f"  Criteria: {evaluation['criteria_match']:.1f}%\n\n")
        
        if evaluation['opportunities']:
            f.write("Opportunities:\n")
            for opp in evaluation['opportunities']:
                f.write(f"  - {opp}\n")
            f.write("\n")
        
        if evaluation['risk_factors']:
            f.write("Risk Factors:\n")
            for risk in evaluation['risk_factors']:
                f.write(f"  - {risk}\n")
        
        f.write("=" * 80 + "\n")
    
    logger.info(f"Decision report saved: {report_path}")

def generate_html_visualizations(extraction_data: Dict, evaluation: Dict, output_dir: str):
    """Generate HTML dashboard."""
    logger.info("Generating HTML visualization")
    
    html_path = os.path.join(output_dir, 'tender_report.html')
    
    html_content = f"""<!DOCTYPE html>
<html>
<head>
    <title>Tender Analysis Report</title>
    <style>
        body {{ font-family: Arial; margin: 20px; background: #f5f5f5; }}
        .container {{ max-width: 1200px; margin: auto; background: white; padding: 20px; border-radius: 8px; }}
        .header {{ text-align: center; border-bottom: 3px solid #2c3e50; padding-bottom: 20px; }}
        h1 {{ color: #2c3e50; }}
        .recommendation {{ padding: 20px; margin: 20px 0; border-radius: 5px; text-align: center; font-size: 1.3em; }}
        .yes {{ background: #d4edda; color: #155724; border: 2px solid #28a745; }}
        .maybe {{ background: #fff3cd; color: #856404; border: 2px solid #ffc107; }}
        .no {{ background: #f8d7da; color: #721c24; border: 2px solid #f5c6cb; }}
        .scores {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 20px; margin: 20px 0; }}
        .score-card {{ padding: 20px; background: linear-gradient(135deg, #667eea, #764ba2); color: white; text-align: center; border-radius: 8px; }}
        .score-card .value {{ font-size: 2.5em; font-weight: bold; }}
        table {{ width: 100%; border-collapse: collapse; margin: 20px 0; }}
        th, td {{ padding: 12px; text-align: left; border-bottom: 1px solid #ddd; }}
        th {{ background: #2c3e50; color: white; }}
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>🏗️ Tender Analysis Report</h1>
            <p>{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        </div>
        
        <div class="recommendation {evaluation['recommendation'].lower()}">
            {'✓ RECOMMENDED' if evaluation['recommendation'] == 'YES' else '⚠ CONDITIONAL' if evaluation['recommendation'] == 'MAYBE' else '✗ NOT RECOMMENDED'}
            <br><span style="font-size: 0.9em;">Score: {evaluation['overall_score']:.1f}%</span>
        </div>
        
        <div class="scores">
            <div class="score-card"><h3>Machinery</h3><div class="value">{evaluation['machinery_match']:.0f}%</div></div>
            <div class="score-card"><h3>Staff</h3><div class="value">{evaluation['staff_match']:.0f}%</div></div>
            <div class="score-card"><h3>Certs</h3><div class="value">{evaluation['certifications_match']:.0f}%</div></div>
            <div class="score-card"><h3>Criteria</h3><div class="value">{evaluation['criteria_match']:.0f}%</div></div>
        </div>
        
        <h2>Machinery Requirements</h2>
        <table>
            <tr><th>Equipment</th><th>Quantity</th><th>Specifications</th></tr>
            {''.join(f"<tr><td>{m['type']}</td><td>{m['quantity']}</td><td>{m['specifications']}</td></tr>" for m in extraction_data.get('machinery', []))}
        </table>
        
        <h2>Staff Requirements</h2>
        <table>
            <tr><th>Position</th><th>Quantity</th><th>Experience</th></tr>
            {''.join(f"<tr><td>{s['position']}</td><td>{s['quantity']}</td><td>{s['experience_years']}y</td></tr>" for s in extraction_data.get('staff', []))}
        </table>
    </div>
</body>
</html>"""
    
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)
    logger.info(f"HTML saved: {html_path}")

print("✓ Report generation functions loaded")

✓ Report generation functions loaded


## 14. Main Analysis Pipeline

Execute complete tender analysis workflow.

In [15]:
# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function."""
    start_time = time.time()
    
    logger.info("=" * 80)
    logger.info("TENDER ANALYSIS SYSTEM v8.0 - STARTING")
    logger.info("=" * 80)
    
    # Step 1: Extract PDF
    logger.info("\n[1/8] Extracting PDF...")
    if not os.path.exists(config.PDF_PATH):
        logger.error(f"PDF not found: {config.PDF_PATH}")
        return
    
    text, page_map = extract_text_from_pdf(config.PDF_PATH)
    if not text:
        logger.error("Failed to extract text")
        return
    
    print(f"✓ Extracted {len(text)} characters from {len(page_map)} pages")
    
    # Step 2: Chunk text
    logger.info("\n[2/8] Chunking text...")
    chunks = smart_chunk_text(text, page_map)
    print(f"✓ Created {len(chunks)} chunks")
    
    # Step 3: Create vector store
    logger.info("\n[3/8] Creating vector store...")
    vector_store = create_vector_store(chunks, model)
    print("✓ Vector store created")
    
    # Step 4: Extract deadline
    logger.info("\n[4/8] Extracting deadline...")
    deadline = extract_deadline(text)
    print(f"✓ Deadline found: {deadline['full'] if deadline['found'] else 'Not found'}")
    
    # Step 5: Extract machinery
    logger.info("\n[5/8] Extracting machinery...")
    machinery = extract_machinery_from_table(text)
    print(f"✓ Found {len(machinery)} machinery items")
    
    # Step 6: Extract staff
    logger.info("\n[6/8] Extracting staff...")
    staff = extract_staff_smart_v2(text, vector_store, model)
    print(f"✓ Found {len(staff)} staff positions")
    
    # Step 7: Extract certifications
    logger.info("\n[7/8] Extracting certifications...")
    certifications = extract_certifications_smart_v2(text, vector_store, model)
    print(f"✓ Found {len(certifications)} certifications")
    
    # Step 8: Match criteria
    logger.info("\n[8/8] Matching criteria...")
    criteria_matching = match_criteria_with_similarity(vector_store, model)
    print(f"✓ Matched {criteria_matching['_statistics']['matched_criteria']}/{criteria_matching['_statistics']['total_criteria']} criteria")
    
    # Compile results
    extraction_data = {
        'metadata': {
            'pdf_file': config.PDF_PATH,
            'extraction_date': datetime.now().isoformat(),
            'total_pages': len(page_map),
            'total_chunks': len(chunks),
            'processing_time_seconds': round(time.time() - start_time, 2),
            'version': '8.0 - Complete'
        },
        'deadline': deadline,
        'machinery': machinery,
        'staff': staff,
        'certifications': certifications,
        'criteria_matching': criteria_matching
    }
    
    # Evaluate
    logger.info("\n[OUTPUT] Evaluating tender...")
    evaluation = evaluate_tender_participation(extraction_data)
    print(f"\n✓ Recommendation: {evaluation['recommendation']}")
    print(f"✓ Overall Score: {evaluation['overall_score']:.1f}%")
    
    # Generate reports
    logger.info("\n[OUTPUT] Generating reports...")
    generate_output_report(extraction_data, config.OUTPUT_DIR)
    generate_tender_decision_report(evaluation, extraction_data, config.OUTPUT_DIR)
    generate_html_visualizations(extraction_data, evaluation, config.OUTPUT_DIR)
    
    # Summary
    logger.info("\n" + "=" * 80)
    logger.info("ANALYSIS COMPLETE")
    logger.info("=" * 80)
    print("\n✓ Reports saved to:", config.OUTPUT_DIR)
    print("  - tender_extraction_report.json")
    print("  - tender_extraction_report.txt")
    print("  - tender_decision_report.txt")
    print("  - tender_report.html")
    print("  - tender_extraction_report.xlsx")
    
    return extraction_data, evaluation

# Run the analysis
print("Starting analysis...")
extraction_data, evaluation = main()
print("\n✓ Analysis completed successfully!")

2025-11-12 06:02:45,503 - INFO - ================================================================================
2025-11-12 06:02:45,504 - INFO - TENDER ANALYSIS SYSTEM v8.0 - STARTING
2025-11-12 06:02:45,505 - INFO - ================================================================================
2025-11-12 06:02:45,506 - INFO - 
[1/8] Extracting PDF...


Starting analysis...


2025-11-12 06:02:45,782 - INFO - Extracted 211700 characters from 117 pages
2025-11-12 06:02:45,783 - INFO - 
[2/8] Chunking text...
2025-11-12 06:02:45,801 - INFO - Created 594 chunks across 117 pages
2025-11-12 06:02:45,803 - INFO - 
[3/8] Creating vector store...
2025-11-12 06:02:45,805 - INFO - Generating embeddings for 594 chunks


✓ Extracted 211700 characters from 117 pages
✓ Created 594 chunks


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

...

✓ Found 1 certifications


2025-11-12 06:03:10,482 - INFO - Processing category: Kritere Financiare
2025-11-12 06:03:10,636 - INFO - Processing category: Kritere të Cilësisë dhe Sigurisë
2025-11-12 06:03:10,756 - INFO - Processing category: Kritere të Vlerësimit të Ofertës
2025-11-12 06:03:10,855 - INFO - Processing category: Kritere për Përgjegjshmëri Sociale & Ligjore
2025-11-12 06:03:10,945 - INFO - Processing category: Kritere të Veçanta
2025-11-12 06:03:11,043 - INFO - Matched 23/27 criteria (85.2%)
2025-11-12 06:03:11,045 - INFO - 
[OUTPUT] Evaluating tender...
2025-11-12 06:03:11,046 - INFO - Evaluating tender participation suitability
2025-11-12 06:03:11,046 - INFO - Evaluation complete - Score: 59.6%, Recommendation: NO
2025-11-12 06:03:11,048 - INFO - 
[OUTPUT] Generating reports...
2025-11-12 06:03:11,049 - INFO - Generating output reports
2025-11-12 06:03:11,051 - INFO - JSON saved: outputs_v8\tender_extraction_report.json
2025-11-12 06:03:11,053 - INFO - TEXT saved: outputs_v8\tender_extraction_repo

✓ Matched 23/27 criteria

✓ Recommendation: NO
✓ Overall Score: 59.6%


2025-11-12 06:03:11,521 - INFO - EXCEL saved: outputs_v8\tender_extraction_report.xlsx
2025-11-12 06:03:11,523 - INFO - Generating decision report
2025-11-12 06:03:11,524 - INFO - Decision report saved: outputs_v8\tender_decision_report.txt
2025-11-12 06:03:11,525 - INFO - Generating HTML visualization
2025-11-12 06:03:11,529 - INFO - HTML saved: outputs_v8\tender_report.html
2025-11-12 06:03:11,530 - INFO - 
2025-11-12 06:03:11,531 - INFO - ANALYSIS COMPLETE
2025-11-12 06:03:11,531 - INFO - ================================================================================



✓ Reports saved to: outputs_v8
  - tender_extraction_report.json
  - tender_extraction_report.txt
  - tender_decision_report.txt
  - tender_report.html
  - tender_extraction_report.xlsx

✓ Analysis completed successfully!


## 15. Optional: Save to Database

Store analysis results in MySQL database.

In [18]:
# ============================================================================
# OPTIONAL: DATABASE INTEGRATION
# ============================================================================

"""
Complete Database Integration Module for Tender Analysis System
Full integration with all 9 tables in tender_database schema
"""

import logging
from datetime import datetime
from typing import Dict, List, Optional, Tuple
import uuid
import json

logger = logging.getLogger(__name__)

# Try to import mysql connector
try:
    import mysql.connector
    from mysql.connector import Error
    MYSQL_AVAILABLE = True
except ImportError:
    MYSQL_AVAILABLE = False
    logger.warning("mysql-connector-python not installed. Install with: pip install mysql-connector-python")


class TenderDatabase:
    """Complete database operations for tender analysis system."""
    
    def __init__(self, host='localhost', user='root', password='', database='tender_database'):
        """Initialize database connection."""
        if not MYSQL_AVAILABLE:
            raise ImportError("mysql-connector-python required. Install: pip install mysql-connector-python")
        
        self.host = host
        self.user = user
        self.password = password
        self.database = database
        self.connection = None
        
    def connect(self) -> bool:
        """Establish database connection."""
        try:
            self.connection = mysql.connector.connect(
                host=self.host,
                user=self.user,
                password=self.password,
                database=self.database,
                autocommit=True
            )
            logger.info(f"✓ Connected to database '{self.database}'")
            return True
        except Error as e:
            logger.error(f"✗ Connection error: {e}")
            return False
    
    def disconnect(self):
        """Close database connection."""
        if self.connection and self.connection.is_connected():
            self.connection.close()
            logger.info("✓ Database disconnected")
    
    # ========================================================================
    # TENDER DOCUMENTS OPERATIONS
    # ========================================================================
    
    def insert_tender_document(self, tender_data: Dict) -> Optional[str]:
        """
        Insert tender document into tender_documents table.
        
        Args:
            tender_data: {
                'tender_number': str (required),
                'tender_title': str (required),
                'issuing_authority': str (required),
                'publication_date': date,
                'submission_deadline': date (required),
                'language': str ('albanian' default),
                'document_path': str,
                'raw_text': str,
                'processed_text': str,
                'status': str ('new', 'processed', 'approved', etc.),
                'estimated_value': float,
                'currency': str ('EUR' default)
            }
        
        Returns:
            str: Tender ID or None
        """
        if not self.connection or not self.connection.is_connected():
            logger.error("Not connected to database")
            return None
        
        try:
            cursor = self.connection.cursor()
            
            tender_id = str(uuid.uuid4())
            
            query = """
            INSERT INTO tender_documents 
            (tender_id, tender_number, tender_title, issuing_authority, publication_date, 
             submission_deadline, language, document_path, raw_text, processed_text, 
             status, estimated_value, currency)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """
            
            values = (
                tender_id,
                tender_data.get('tender_number', f'TENDER-{tender_id[:8]}'),
                tender_data.get('tender_title', 'Untitled'),
                tender_data.get('issuing_authority', 'Unknown'),
                tender_data.get('publication_date', datetime.now().date()),
                tender_data.get('submission_deadline'),
                tender_data.get('language', 'albanian'),
                tender_data.get('document_path', ''),
                tender_data.get('raw_text', '')[:5000],
                tender_data.get('processed_text', '')[:5000],
                tender_data.get('status', 'new'),
                tender_data.get('estimated_value'),
                tender_data.get('currency', 'EUR')
            )
            
            cursor.execute(query, values)
            self.connection.commit()
            
            logger.info(f"✓ Tender inserted: {tender_id}")
            return tender_id
            
        except Error as e:
            logger.error(f"✗ Error inserting tender: {e}")
            return None
        finally:
            if cursor:
                cursor.close()
    
    def update_tender_status(self, tender_id: str, status: str, processed_text: str = None) -> bool:
        """Update tender status and optionally processed text."""
        if not self.connection or not self.connection.is_connected():
            return False
        
        try:
            cursor = self.connection.cursor()
            
            if processed_text:
                query = """
                UPDATE tender_documents
                SET status = %s, processed_text = %s, processed_at = NOW()
                WHERE tender_id = %s
                """
                values = (status, processed_text[:5000], tender_id)
            else:
                query = """
                UPDATE tender_documents
                SET status = %s, processed_at = NOW()
                WHERE tender_id = %s
                """
                values = (status, tender_id)
            
            cursor.execute(query, values)
            self.connection.commit()
            
            logger.info(f"✓ Tender {tender_id} status updated to: {status}")
            return True
            
        except Error as e:
            logger.error(f"✗ Error updating tender: {e}")
            return False
        finally:
            if cursor:
                cursor.close()
    
    def get_tender(self, tender_id: str) -> Optional[Dict]:
        """Retrieve single tender document."""
        if not self.connection or not self.connection.is_connected():
            return None
        
        try:
            cursor = self.connection.cursor(dictionary=True)
            query = "SELECT * FROM tender_documents WHERE tender_id = %s"
            cursor.execute(query, (tender_id,))
            result = cursor.fetchone()
            
            if result:
                logger.info(f"✓ Retrieved tender: {tender_id}")
            return result
            
        except Error as e:
            logger.error(f"✗ Error retrieving tender: {e}")
            return None
        finally:
            if cursor:
                cursor.close()
    
    def get_all_tenders(self, status: str = None, limit: int = 100) -> List[Dict]:
        """Retrieve tenders, optionally filtered by status."""
        if not self.connection or not self.connection.is_connected():
            return []
        
        try:
            cursor = self.connection.cursor(dictionary=True)
            
            if status:
                query = """
                SELECT * FROM tender_documents 
                WHERE status = %s 
                ORDER BY created_at DESC 
                LIMIT %s
                """
                cursor.execute(query, (status, limit))
            else:
                query = """
                SELECT * FROM tender_documents 
                ORDER BY created_at DESC 
                LIMIT %s
                """
                cursor.execute(query, (limit,))
            
            results = cursor.fetchall()
            logger.info(f"✓ Retrieved {len(results)} tenders")
            return results
            
        except Error as e:
            logger.error(f"✗ Error retrieving tenders: {e}")
            return []
        finally:
            if cursor:
                cursor.close()
    
    # ========================================================================
    # TENDER REQUIREMENTS OPERATIONS
    # ========================================================================
    
    def insert_requirement(self, requirement_data: Dict) -> Optional[str]:
        """
        Insert tender requirement.
        
        Args:
            requirement_data: {
                'tender_id': str (required),
                'requirement_type': str (MACHINERY, STAFF, CERTIFICATION),
                'requirement_category': str,
                'requirement_text': str (required),
                'extracted_text': str,
                'parsed_requirements': dict,
                'priority': str (low, medium, high),
                'is_mandatory': bool (default True)
            }
        
        Returns:
            str: Requirement ID or None
        """
        if not self.connection or not self.connection.is_connected():
            return None
        
        try:
            cursor = self.connection.cursor()
            
            req_id = str(uuid.uuid4())
            
            query = """
            INSERT INTO tender_requirements
            (requirement_id, tender_id, requirement_type, requirement_category,
             requirement_text, extracted_text, parsed_requirements, priority, is_mandatory)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
            """
            
            parsed = json.dumps(requirement_data.get('parsed_requirements', {}))
            
            values = (
                req_id,
                requirement_data.get('tender_id'),
                requirement_data.get('requirement_type', 'GENERAL'),
                requirement_data.get('requirement_category', 'general'),
                requirement_data.get('requirement_text', ''),
                requirement_data.get('extracted_text', ''),
                parsed,
                requirement_data.get('priority', 'medium'),
                requirement_data.get('is_mandatory', True)
            )
            
            cursor.execute(query, values)
            self.connection.commit()
            
            logger.info(f"✓ Requirement inserted: {req_id}")
            return req_id
            
        except Error as e:
            logger.error(f"✗ Error inserting requirement: {e}")
            return None
        finally:
            if cursor:
                cursor.close()
    
    def insert_requirements_batch(self, tender_id: str, requirements: List[Dict]) -> Tuple[int, int]:
        """
        Insert multiple requirements at once.
        
        Returns:
            Tuple: (successful_count, failed_count)
        """
        if not self.connection or not self.connection.is_connected():
            return (0, len(requirements))
        
        success = 0
        failed = 0
        
        for req in requirements:
            req['tender_id'] = tender_id
            if self.insert_requirement(req):
                success += 1
            else:
                failed += 1
        
        logger.info(f"✓ Batch insert: {success} successful, {failed} failed")
        return (success, failed)
    
    def get_tender_requirements(self, tender_id: str, req_type: str = None) -> List[Dict]:
        """Get requirements for tender, optionally filtered by type."""
        if not self.connection or not self.connection.is_connected():
            return []
        
        try:
            cursor = self.connection.cursor(dictionary=True)
            
            if req_type:
                query = """
                SELECT * FROM tender_requirements 
                WHERE tender_id = %s AND requirement_type = %s
                ORDER BY priority DESC
                """
                cursor.execute(query, (tender_id, req_type))
            else:
                query = """
                SELECT * FROM tender_requirements 
                WHERE tender_id = %s
                ORDER BY requirement_type, priority DESC
                """
                cursor.execute(query, (tender_id,))
            
            results = cursor.fetchall()
            logger.info(f"✓ Retrieved {len(results)} requirements for tender {tender_id}")
            return results
            
        except Error as e:
            logger.error(f"✗ Error retrieving requirements: {e}")
            return []
        finally:
            if cursor:
                cursor.close()
    
    # ========================================================================
    # MACHINERY OPERATIONS
    # ========================================================================
    
    def insert_machinery(self, machinery_data: Dict) -> Optional[str]:
        """
        Insert machinery into fleet.
        
        Args:
            machinery_data: {
                'machinery_name': str (required),
                'machinery_type': str (required),
                'model': str,
                'serial_number': str,
                'manufacturer': str,
                'year_manufactured': int,
                'capacity': str,
                'specifications': dict,
                'location': str,
                'purchase_date': date,
                'maintenance_cost': float
            }
        
        Returns:
            str: Machinery ID or None
        """
        if not self.connection or not self.connection.is_connected():
            return None
        
        try:
            cursor = self.connection.cursor()
            
            mach_id = str(uuid.uuid4())
            specs = json.dumps(machinery_data.get('specifications', {}))
            
            query = """
            INSERT INTO machinery
            (machinery_id, machinery_name, machinery_type, model, serial_number,
             manufacturer, year_manufactured, capacity, specifications, location,
             purchase_date, maintenance_cost)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """
            
            values = (
                mach_id,
                machinery_data.get('machinery_name'),
                machinery_data.get('machinery_type'),
                machinery_data.get('model', ''),
                machinery_data.get('serial_number', ''),
                machinery_data.get('manufacturer', ''),
                machinery_data.get('year_manufactured'),
                machinery_data.get('capacity', ''),
                specs,
                machinery_data.get('location', ''),
                machinery_data.get('purchase_date'),
                machinery_data.get('maintenance_cost')
            )
            
            cursor.execute(query, values)
            self.connection.commit()
            
            logger.info(f"✓ Machinery inserted: {mach_id}")
            return mach_id
            
        except Error as e:
            logger.error(f"✗ Error inserting machinery: {e}")
            return None
        finally:
            if cursor:
                cursor.close()
    
    def get_available_machinery(self, machinery_type: str = None) -> List[Dict]:
        """Get available machinery from fleet."""
        if not self.connection or not self.connection.is_connected():
            return []
        
        try:
            cursor = self.connection.cursor(dictionary=True)
            
            if machinery_type:
                query = """
                SELECT * FROM machinery
                WHERE current_status = 'available' AND machinery_type = %s
                ORDER BY machinery_name
                """
                cursor.execute(query, (machinery_type,))
            else:
                query = """
                SELECT * FROM machinery
                WHERE current_status = 'available'
                ORDER BY machinery_type, machinery_name
                """
                cursor.execute(query)
            
            results = cursor.fetchall()
            logger.info(f"✓ Retrieved {len(results)} available machinery")
            return results
            
        except Error as e:
            logger.error(f"✗ Error retrieving machinery: {e}")
            return []
        finally:
            if cursor:
                cursor.close()
    
    # ========================================================================
    # EMPLOYEES OPERATIONS
    # ========================================================================
    
    def insert_employee(self, employee_data: Dict) -> Optional[str]:
        """
        Insert employee record.
        
        Args:
            employee_data: {
                'first_name': str (required),
                'last_name': str (required),
                'email': str (required),
                'phone': str,
                'position': str (required),
                'department': str,
                'specialization': list,
                'years_experience': int,
                'certifications': list,
                'languages_spoken': list,
                'hire_date': date (required),
                'salary': float,
                'availability_start_date': date,
                'availability_end_date': date
            }
        
        Returns:
            str: Employee ID or None
        """
        if not self.connection or not self.connection.is_connected():
            return None
        
        try:
            cursor = self.connection.cursor()
            
            emp_id = str(uuid.uuid4())
            spec = json.dumps(employee_data.get('specialization', []))
            certs = json.dumps(employee_data.get('certifications', []))
            langs = json.dumps(employee_data.get('languages_spoken', []))
            
            query = """
            INSERT INTO employees
            (employee_id, first_name, last_name, email, phone, position, department,
             specialization, years_experience, certifications, languages_spoken,
             hire_date, salary, availability_start_date, availability_end_date)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """
            
            values = (
                emp_id,
                employee_data.get('first_name'),
                employee_data.get('last_name'),
                employee_data.get('email'),
                employee_data.get('phone', ''),
                employee_data.get('position'),
                employee_data.get('department', ''),
                spec,
                employee_data.get('years_experience', 0),
                certs,
                langs,
                employee_data.get('hire_date', datetime.now().date()),
                employee_data.get('salary'),
                employee_data.get('availability_start_date'),
                employee_data.get('availability_end_date')
            )
            
            cursor.execute(query, values)
            self.connection.commit()
            
            logger.info(f"✓ Employee inserted: {emp_id}")
            return emp_id
            
        except Error as e:
            logger.error(f"✗ Error inserting employee: {e}")
            return None
        finally:
            if cursor:
                cursor.close()
    
    def get_available_employees(self, position: str = None) -> List[Dict]:
        """Get available employees."""
        if not self.connection or not self.connection.is_connected():
            return []
        
        try:
            cursor = self.connection.cursor(dictionary=True)
            
            if position:
                query = """
                SELECT * FROM employees
                WHERE current_status = 'available' AND position = %s
                ORDER BY last_name, first_name
                """
                cursor.execute(query, (position,))
            else:
                query = """
                SELECT * FROM employees
                WHERE current_status = 'available'
                ORDER BY position, last_name, first_name
                """
                cursor.execute(query)
            
            results = cursor.fetchall()
            logger.info(f"✓ Retrieved {len(results)} available employees")
            return results
            
        except Error as e:
            logger.error(f"✗ Error retrieving employees: {e}")
            return []
        finally:
            if cursor:
                cursor.close()
    
    # ========================================================================
    # PROJECTS OPERATIONS
    # ========================================================================
    
    def insert_project(self, project_data: Dict) -> Optional[str]:
        """Insert project record."""
        if not self.connection or not self.connection.is_connected():
            return None
        
        try:
            cursor = self.connection.cursor()
            
            proj_id = str(uuid.uuid4())
            
            query = """
            INSERT INTO projects
            (project_id, project_name, project_type, client_name, start_date,
             end_date, estimated_end_date, budget, actual_cost, status, 
             description, location, project_manager_id)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """
            
            values = (
                proj_id,
                project_data.get('project_name'),
                project_data.get('project_type'),
                project_data.get('client_name', ''),
                project_data.get('start_date', datetime.now().date()),
                project_data.get('end_date'),
                project_data.get('estimated_end_date'),
                project_data.get('budget'),
                project_data.get('actual_cost'),
                project_data.get('status', 'planning'),
                project_data.get('description', ''),
                project_data.get('location', ''),
                project_data.get('project_manager_id')
            )
            
            cursor.execute(query, values)
            self.connection.commit()
            
            logger.info(f"✓ Project inserted: {proj_id}")
            return proj_id
            
        except Error as e:
            logger.error(f"✗ Error inserting project: {e}")
            return None
        finally:
            if cursor:
                cursor.close()
    
    # ========================================================================
    # COMPLIANCE & ANALYTICS
    # ========================================================================
    
    def get_tender_summary(self) -> Dict:
        """Get summary statistics for all tenders."""
        if not self.connection or not self.connection.is_connected():
            return {}
        
        try:
            cursor = self.connection.cursor(dictionary=True)
            
            # Count by status
            query = """
            SELECT status, COUNT(*) as count
            FROM tender_documents
            GROUP BY status
            """
            cursor.execute(query)
            status_counts = {row['status']: row['count'] for row in cursor.fetchall()}
            
            # Total estimated value
            query = """
            SELECT 
                COUNT(*) as total_tenders,
                SUM(estimated_value) as total_value,
                AVG(estimated_value) as avg_value
            FROM tender_documents
            WHERE estimated_value IS NOT NULL
            """
            cursor.execute(query)
            value_stats = cursor.fetchone()
            
            summary = {
                'status_breakdown': status_counts,
                'total_tenders': value_stats['total_tenders'],
                'total_estimated_value': value_stats['total_value'],
                'average_tender_value': value_stats['avg_value']
            }
            
            logger.info("✓ Retrieved tender summary")
            return summary
            
        except Error as e:
            logger.error(f"✗ Error retrieving summary: {e}")
            return {}
        finally:
            if cursor:
                cursor.close()
    
    def get_requirements_summary(self, tender_id: str) -> Dict:
        """Get summary of requirements for a tender."""
        if not self.connection or not self.connection.is_connected():
            return {}
        
        try:
            cursor = self.connection.cursor(dictionary=True)
            
            query = """
            SELECT 
                requirement_type,
                COUNT(*) as count,
                SUM(CASE WHEN is_mandatory THEN 1 ELSE 0 END) as mandatory_count
            FROM tender_requirements
            WHERE tender_id = %s
            GROUP BY requirement_type
            """
            cursor.execute(query, (tender_id,))
            results = cursor.fetchall()
            
            summary = {row['requirement_type']: {
                'total': row['count'],
                'mandatory': row['mandatory_count']
            } for row in results}
            
            logger.info("✓ Retrieved requirements summary")
            return summary
            
        except Error as e:
            logger.error(f"✗ Error retrieving requirements summary: {e}")
            return {}
        finally:
            if cursor:
                cursor.close()


def save_complete_tender_analysis(pdf_path: str, extraction_data: Dict, evaluation: Dict,
                                  db_host='localhost', db_user='root', db_password='') -> Optional[str]:
    """
    Complete workflow: Save entire tender analysis to database.
    
    Args:
        pdf_path: Path to tender PDF
        extraction_data: Extracted requirements and data
        evaluation: Evaluation results
        
    Returns:
        str: Tender ID if successful, None otherwise
    """
    logger.info("\n" + "="*80)
    logger.info("COMPLETE TENDER ANALYSIS DATABASE SAVE")
    logger.info("="*80)
    
    db = TenderDatabase(host=db_host, user=db_user, password=db_password)
    
    if not db.connect():
        logger.error("✗ Failed to connect to database")
        return None
    
    try:
        # 1. Insert Tender Document
        tender_data = {
            'tender_number': pdf_path.split('/')[-1],
            'tender_title': extraction_data.get('metadata', {}).get('pdf_file', 'Tender'),
            'issuing_authority': 'Extracted from PDF',
            'submission_deadline': extraction_data.get('deadline', {}).get('date'),
            'language': 'albanian',
            'status': 'processed',
            'raw_text': 'PDF extracted',
            'estimated_value': extraction_data.get('estimated_value'),
            'currency': 'EUR'
        }
        
        tender_id = db.insert_tender_document(tender_data)
        if not tender_id:
            logger.error("✗ Failed to create tender record")
            return None
        
        # 2. Insert Machinery Requirements
        machinery = extraction_data.get('machinery', [])
        if machinery:
            for mach in machinery:
                req_data = {
                    'tender_id': tender_id,
                    'requirement_type': 'MACHINERY',
                    'requirement_category': 'equipment',
                    'requirement_text': f"{mach.get('quantity')}x {mach.get('type')}",
                    'parsed_requirements': mach,
                    'is_mandatory': True
                }
                db.insert_requirement(req_data)
            logger.info(f"✓ Inserted {len(machinery)} machinery requirements")
        
        # 3. Insert Staff Requirements
        staff = extraction_data.get('staff', [])
        if staff:
            for s in staff:
                req_data = {
                    'tender_id': tender_id,
                    'requirement_type': 'STAFF',
                    'requirement_category': 'personnel',
                    'requirement_text': f"{s.get('quantity')}x {s.get('position')}",
                    'parsed_requirements': s,
                    'is_mandatory': True
                }
                db.insert_requirement(req_data)
            logger.info(f"✓ Inserted {len(staff)} staff requirements")
        
        # 4. Insert Certifications
        certs = extraction_data.get('certifications', [])
        if certs:
            for cert in certs:
                req_data = {
                    'tender_id': tender_id,
                    'requirement_type': 'CERTIFICATION',
                    'requirement_category': 'compliance',
                    'requirement_text': cert,
                    'is_mandatory': True
                }
                db.insert_requirement(req_data)
            logger.info(f"✓ Inserted {len(certs)} certification requirements")
        
        # 5. Update Tender Status with Evaluation
        status = 'RECOMMENDED' if evaluation.get('recommendation') == 'YES' else \
                 'CONDITIONAL' if evaluation.get('recommendation') == 'MAYBE' else \
                 'NOT_RECOMMENDED'
        
        processed_text = f"Score: {evaluation.get('overall_score', 0):.1f}% - {evaluation.get('recommendation')}"
        db.update_tender_status(tender_id, status, processed_text)
        
        # Print Summary
        print("\n" + "="*80)
        print("✓ TENDER ANALYSIS SAVED TO DATABASE")
        print("="*80)
        print(f"Tender ID:        {tender_id}")
        print(f"Status:           {status}")
        print(f"Recommendation:   {evaluation.get('recommendation')}")
        print(f"Score:            {evaluation.get('overall_score', 0):.1f}%")
        print(f"Machinery Items:  {len(machinery)}")
        print(f"Staff Positions:  {len(staff)}")
        print(f"Certifications:   {len(certs)}")
        
        print("\nDatabase Summary:")
        summary = db.get_requirements_summary(tender_id)
        for req_type, counts in summary.items():
            print(f"  {req_type}: {counts['total']} total, {counts['mandatory']} mandatory")
        
        print("\nQuery Examples:")
        print(f"  SELECT * FROM tender_documents WHERE tender_id = '{tender_id}';")
        print(f"  SELECT * FROM tender_requirements WHERE tender_id = '{tender_id}';")
        print("="*80 + "\n")
        
        return tender_id
        
    except Exception as e:
        logger.error(f"✗ Error: {e}")
        return None
    finally:
        db.disconnect()


if __name__ == "__main__":
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )
    print("✓ Complete database module loaded successfully")

✓ Complete database module loaded successfully
